<a href="https://colab.research.google.com/github/Aisyaaswy/Pemrograman_Mesin/blob/main/jobsheet_6/dataset_pbl/HNSW/praktikum_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **PRAKTIKUM 3**


In [1]:
!pip install hnswlib

  Installing build dependencies ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679800 sha256=91df27181ee4c1ac244aad3015fdb1a489ce588823f089e7f06699992c717d9f
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Successfully built hnswlib
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679800 sha256=91df27181ee4c1ac244aad3015fdb1a489ce588823f089e7f06699992c717d9f
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Successfully built hnswlib


Percobaan berikut akan membandingkan exact NN dengan HNSW pada 1000 data 2D.

In [3]:
import hnswlib
import numpy as np
import pandas as pd
import time
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# ===========================
# 1. Load & Standarisasi Dataset Sleep Health (2 Fitur)
# ===========================
df = pd.read_csv('Sleep_health_and_lifestyle_dataset.csv')
fitur_2d = ['Sleep Duration', 'Stress Level']
data = StandardScaler().fit_transform(df[fitur_2d]).astype(np.float32)

num_elements, dim = data.shape

# Query point (ambil data baris ke-0)
query = data[0:1].copy()
k = 5   # cari 5 tetangga terdekat

# ===========================
# 2. Exact NN (Brute Force)
# ===========================
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(data)

start = time.time()
distances, indices = nn.kneighbors(query)
end = time.time()

print("=== Exact NN ===")
print("Indices:", indices)
print("Distances:", distances)
print("Waktu:", end - start, "detik")

# ===========================
# 3. HNSW
# ===========================
p = hnswlib.Index(space='l2', dim=dim)
p.init_index(max_elements=num_elements, ef_construction=100, M=16)
p.add_items(data)
p.set_ef(50)

start = time.time()
labels, distances = p.knn_query(query, k=k)
end = time.time()

print("\n=== HNSW ===")
print("Indices:", labels)
print("Distances:", distances)
print("Waktu:", end - start, "detik")

=== Exact NN ===
Indices: [[  0 106  69  68   6]]
Distances: [[0.         0.         0.12585068 0.12585068 0.61787677]]
Waktu: 0.01993536949157715 detik

=== HNSW ===
Indices: [[  0 106  68  69   6]]
Distances: [[0.         0.         0.01583839 0.01583839 0.3817717 ]]
Waktu: 0.00014829635620117188 detik


Lakukan percobaan pada metric distance yang berbeda, 374 vs 1jt data, 2D vs 5D data. catat hasilnya pada tabel yang anda buat sendiri seperti pada praktikum 1.

In [4]:
# Siapkan 5 fitur numerik untuk eksperimen 5D
fitur_5d = ['Age', 'Sleep Duration', 'Quality of Sleep', 'Stress Level', 'Heart Rate']
data_5d = StandardScaler().fit_transform(df[fitur_5d]).astype(np.float32)

# Konfigurasi: (Metric, Jumlah Data, Dimensi)
configs = [
    ('l2', 374, 2),
    ('l2', 1000000, 2),
    ('l2', 374, 5),
    ('l2', 1000000, 5),
    ('ip', 374, 2),
    ('ip', 1000000, 2),
    ('ip', 374, 5),
    ('ip', 1000000, 5)
]

hasil_eksperimen = []
k = 5
np.random.seed(42)

for space_metric, num_elements, dim in configs:
    base_data = data if dim == 2 else data_5d

    # 1. Siapkan Data (tambahkan sedikit variasi acak pada simulasi 1 juta data agar tidak kembar identik)
    if num_elements == 374:
        X_exp = base_data.copy()
    else:
        repeats = int(np.ceil(num_elements / len(base_data)))
        X_exp = np.tile(base_data, (repeats, 1))[:num_elements].copy()
        X_exp += np.random.normal(0, 0.01, X_exp.shape).astype(np.float32)

    query = X_exp[0:1].copy()

    # 2. Exact NN (Brute Force)
    start_exact = time.time()
    if space_metric == 'l2':
        exact_distances = np.sum((X_exp - query[0]) ** 2, axis=1)
    else:
        exact_distances = 1.0 - np.dot(X_exp, query[0])
    exact_indices = np.argsort(exact_distances)[:k]
    time_exact = time.time() - start_exact

    # 3. HNSW (Approximate NN) - ef_construction=50 agar cepat di 1 juta data
    p = hnswlib.Index(space=space_metric, dim=dim)
    p.init_index(max_elements=num_elements, ef_construction=50, M=16)
    p.add_items(X_exp, num_threads=4)
    p.set_ef(50)

    start_hnsw = time.time()
    labels, distances = p.knn_query(query, k=k)
    time_hnsw = time.time() - start_hnsw

    # 4. Format hasil index
    str_idx_exact = "[" + " ".join(map(str, exact_indices)) + "]"
    str_idx_hnsw = "[" + " ".join(map(str, labels[0])) + "]"
    metric_label = 'Euclidean (L2)' if space_metric == 'l2' else 'Inner Product (IP)'

    hasil_eksperimen.append({
        'Metric Distance': metric_label,
        'Dimensi': f"{dim}D",
        'Jumlah Data': num_elements,
        'Hasil Index terdekat (ENN vs HNSW)': f"{str_idx_exact} , {str_idx_hnsw}",
        'Waktu komputasi (ENN vs HNSW) detik': f"{time_exact:.5f} , {time_hnsw:.5f}"
    })

# 5. Cetak Tabel Hasil
df_hasil = pd.DataFrame(hasil_eksperimen)
pd.set_option('display.max_colwidth', None)
display(df_hasil)

,Metric Distance,Dimensi,Jumlah Data,Hasil Index terdekat (ENN vs HNSW),Waktu komputasi (ENN vs HNSW) detik
0,Euclidean (L2),2D,374,"[0 106 68 69 222] , [0 106 68 69 231]","0.00016 , 0.00005"
1,Euclidean (L2),2D,1000000,"[0 334356 182138 304168 524080] , [0 334356 182138 304168 524080]","0.08081 , 0.00008"
2,Euclidean (L2),5D,374,"[0 69 68 106 31] , [0 68 69 106 31]","0.00014 , 0.00006"
3,Euclidean (L2),5D,1000000,"[0 600644 40392 289102 63206] , [0 600644 40392 289102 63206]","0.09142 , 0.00014"
4,Inner Product (IP),2D,374,"[81 80 3 4 5] , [80 81 4 5 265]","0.00017 , 0.00006"
5,Inner Product (IP),2D,1000000,"[78994 865142 579032 605586 243929] , [78994 579032 243929 672907 680013]","0.13546 , 0.00012"
6,Inner Product (IP),5D,374,"[3 4 5 81 80] , [3 4 5 80 81]","0.00012 , 0.00007"
7,Inner Product (IP),5D,1000000,"[819811 129408 636552 298082 263675] , [819811 129408 636552 298082 263675]","0.15100 , 0.00007"
